<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/src/Topic_Clustring_and_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalations and Imports

In [35]:
try:
    # Running inside Google Colab → use !pip
    import google.colab
    print("Running in Colab → installing packages...")
    !pip install -q bertopic
    !pip install -q tweet-preprocessor
    !pip install -q spacy-entity-linker
    !pip install -q emoji
    !pip install sentence-transformers scikit-learn
    !pip install "transformers>=4.40.0" sentencepiece accelerate torch
except ImportError:
    # Not running in Colab → do nothing
    pass

In [36]:
import pandas as pd
import matplotlib.pyplot as plt

import re
import emoji

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import spacy

from tqdm.auto import tqdm

# Load Data
[Labeled Datasets for Research on Information Operations](https://zenodo.org/records/14189193)

In [37]:
from pathlib import Path

dataset_name = "Labeled_Datasets/Ecuador"
file_name = "Ecuador_part_3.gzip.parquet"

# Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path = Path("/content/drive/Shareddrives/KeyBoostAuthor") / dataset_name / file_name
    in_colab = True
    print("Running inside Colab. Loading from Google Drive...")

# BGU Slurm Cluster
except ImportError:
    # working directory is: Backbone-Graph/src
    file_path = Path("../data") / dataset_name / file_name
    in_colab = False
    print("Running on Slurm cluster. Loading local file...")


# Load the parquet file
df = pd.read_parquet(file_path)

# for testing
# df = df.head(500).copy() 

Running on Slurm cluster. Loading local file...


In [ ]:
# output_path

file_path = Path(file_path)


if in_colab:
  output_path = (file_path.parents[2] / "Labeled_Dataset_with_Topic_NER" / file_path.parent.name / file_path.name)
else:
  # working directory is: Backbone-Graph/src
  output_folder_name = file_name.stem.stem # file_name_no_suffixes
  output_folder_path = Path("../results") / dataset_name / output_folder_name
  # Example: "../results/Labeled_Datasets/Ecuador/Ecuador_part_3/"


output_folder_path.mkdir(parents=True, exist_ok=True) # Creates the parent directory if needed

In [38]:
df.head()

,postid,post_text,application_name,post_language,in_reply_to_postid,in_reply_to_accountid,post_time,accountid,account_profile_description,follower_count,following_count,account_creation_date,is_repost,reposted_accountid,reposted_postid,hashtags,urls,account_mentions,is_control
100000,43394c24c54037f763e8318026ffba9aeb9317f672,"Queda prohibido no sonreir a los problemas, no...",2727201c363b6fc4afe259ba4094ae0c49cb4df224,None,None,None,2013-02-21 16:02:00,71cc57155d061d447c2c65756c7dfd6873e403a097,🍃🧚‍♀️🧜‍♀️👑,3618,2055,2009-12-22,True,5ff80261b935621244fb38fff05231bf108ba8a488,4af2ce0fdafbc22f1a0be317a2f1025f2b18983194,[],[],[5ff80261b935621244fb38fff05231bf108ba8a488],False
100001,1b6cb8bafe65e8b9e639b7e52075c8ebf34d854d6c,"Te encontraras con gente hipócrita, con amigos...",62010408238d4a0f958ec97eec3db0a4f69bcd0d11,es,None,None,2013-02-21 16:04:04,8949eab1b3f746ed66a069ae5d6c415a57dc546ffc,Agradecida con Dios y con la Vida. Ahora en #...,471,233,2010-08-25,False,None,None,[],[],[],True
100002,4b69e40a8e23031f7ddc3832dfe5f289e851b56299,Aventarle un diccionario a alguien en la cabez...,62010408238d4a0f958ec97eec3db0a4f69bcd0d11,es,None,None,2013-02-21 16:04:12,8949eab1b3f746ed66a069ae5d6c415a57dc546ffc,Agradecida con Dios y con la Vida. Ahora en #...,471,233,2010-08-25,False,None,None,[],[],[],True
100003,f810b69a925e539fb18b86ad0d227751cf072738bd,"Suena mi alarma del celu, tendre que levantarm...",62010408238d4a0f958ec97eec3db0a4f69bcd0d11,es,None,None,2013-02-21 16:04:16,8949eab1b3f746ed66a069ae5d6c415a57dc546ffc,Agradecida con Dios y con la Vida. Ahora en #...,471,233,2010-08-25,True,7e6b7dd1aa865b143fc37f4181e392613d1e6dfe3f,e6e93d6dcfdca0ea6ba2104dc6532ade3f85263147,[],[],[7e6b7dd1aa865b143fc37f4181e392613d1e6dfe3f],True
100004,94e00228f2ac99ef8ae06ab1c5419e74dfd7dc4699,En el mar la vida es mas sabrosa 8' @5b3c3e3a2...,e278f5ccf53957cccf9e97b2ae3fec72d5b892f850,None,None,None,2013-02-21 16:15:00,4a08f9f3e1b73d62e982f899cdfb644e2101bd28a4,Grateful💕,587,270,2010-06-06,True,ad1f5918aa283b31785d589c2ade1c1ca176a23a47,4e9a2c4ec3f5bb6ec24e2936e287ac2fa33b30245d,[],[https://f87702c9a584719155693093bf96b50d218e8...,"[ad1f5918aa283b31785d589c2ade1c1ca176a23a47, 5...",False


## Dataset Statistics

In [39]:
total_posts = len(df)
hashtag_posts = df[df['hashtags'].notna() & (df['hashtags'].str.len() > 0)]
mention_posts = df[df['account_mentions'].notna() & (df['account_mentions'].str.len() > 0)]
url_posts = df[df['urls'].notna() & (df['urls'].str.len() > 0)]

print(f"{'df shape:':<40} {df.shape}")
print(f"{'df posts with hashtags # shape:':<40} {hashtag_posts.shape}  ({len(hashtag_posts)/total_posts*100:.2f}%)")
print(f"{'df posts with account_mentions @ shape:':<40} {mention_posts.shape}  ({len(mention_posts)/total_posts*100:.2f}%)")
print(f"{'df posts with urls shape:':<40} {url_posts.shape}  ({len(url_posts)/total_posts*100:.2f}%)")


print()
print()

print(f"{'**collum name**':<30} **value type**")
for col in df.columns:
    dtype_str = str(df[col].dtype)  # Convert dtype to string
    # get a sample non-null value if exists
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")


print()
print()
print(df['post_language'].value_counts())

df shape:                                (50000, 19)
df posts with hashtags # shape:          (22419, 19)  (44.84%)
df posts with account_mentions @ shape:  (31089, 19)  (62.18%)
df posts with urls shape:                (18968, 19)  (37.94%)


**collum name**                **value type**
postid                         | str
post_text                      | str
application_name               | str
post_language                  | str
in_reply_to_postid             | str
in_reply_to_accountid          | str
post_time                      | Timestamp
accountid                      | str
account_profile_description    | str
follower_count                 | int64
following_count                | int64
account_creation_date          | date
is_repost                      | bool
reposted_accountid             | str
reposted_postid                | str
hashtags                       | ndarray
urls                           | ndarray
account_mentions               | ndarray
is_control          

# Clean Text

In [40]:
def clean_url(url_match):

    url = url_match.group(0)

    # 1. Remove protocol (http:// or https://)
    url = re.sub(r"https?://", "", url)
    url = re.sub(r"www\.", "", url)

    # 2. Remove query parameters (?x=1)
    url = url.split("?")[0]

    # 3. Split into components by "/"
    parts = url.split("/")

    # Domain (first part)
    domain = parts[0].replace(".", "_")  # twitter.com → twitter_com

    # First section (second part of the URL)
    if len(parts) > 1 and parts[1].strip() != "":
        section = parts[1].replace(".", "_")
        token = f" URL_{domain}_{section} "
    else:
        token = f" URL_{domain} "

    return token


In [41]:
def clean_text(text: str) -> str:

    # lowercase
    text = str(text).lower()

    # URLs → domian + first section
    text = re.sub(r"(https?://\S+|www\.\S+)", clean_url, text)

    # Hashtags → HASHTAG_xxx
    text = re.sub(r"#(\w+)", r" HASHTAG_\1 ", text)

    # Mentions → MENTION_xxx
    text = re.sub(r"@(\w+)", r" MENTION_\1 ", text)

    # emoji → EMOJI_xxx
    text = emoji.demojize(text, delimiters=(" EMOJI_", " "))

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [42]:
df["clean_post_text"] = df["post_text"].apply(clean_text)

# Translation to English

In [43]:
# map post_language codes to NLLB-200 language codes
# https://dl-translate.readthedocs.io/en/latest/available_languages/

def map_to_nllb_code(lang_code: str) -> str:
    """
    Map your dataset's post_language codes to NLLB-200 language codes.
    If not in mup return the same language code.
    Args:
        lang_code (str): Post language code.
    Returns:
        str: NLLB-200 language code.
    """
    nlln_mapping = {
    # Europe
    "en": "eng_Latn",   # English
    "es": "spa_Latn",   # Spanish
    "pt": "por_Latn",   # Portuguese
    "fr": "fra_Latn",   # French
    "de": "deu_Latn",   # German
    "it": "ita_Latn",   # Italian
    "nl": "nld_Latn",   # Dutch
    "ru": "rus_Cyrl",   # Russian
    "pl": "pol_Latn",   # Polish
    "sv": "swe_Latn",   # Swedish
    "ca": "cat_Latn",    # Catalan
    "cy": "cym_Latn",    # Welsh
    "et": "est_Latn",    # Estonian

    # Middle East
    "ar": "arb_Arab",   # Arabic (MSA)
    "he": "heb_Hebr",   # Hebrew
    "tr": "tur_Latn",   # Turkish
    "fa": "pes_Arab",   # Persian/Farsi

    # South Asia
    "hi": "hin_Deva",   # Hindi
    "ur": "urd_Arab",   # Urdu
    "bn": "ben_Beng",   # Bengali

    # East Asia
    "zh": "zho_Hans",   # Chinese Simplified
    "ja": "jpn_Jpan",   # Japanese
    "ko": "kor_Hang",   # Korean

    # Southeast Asia
    "vi": "vie_Latn",   # Vietnamese
    "id": "ind_Latn",   # Indonesian (modern code)
    "in": "ind_Latn",   # Indonesian (legacy ISO code)
    "tl": "tgl_Latn",   # Tagalog (Philippines)

    # Africa
    "sw": "swh_Latn",   # Swahili

    # Americas
    "qu": "quy_Latn",   # Quechua (Ayacucho)
}


    if lang_code in nlln_mapping:
        return nlln_mapping[lang_code]
    else:
        return lang_code

In [44]:
df["post_language"] = df["post_language"].apply(map_to_nllb_code)

In [45]:
# load models

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

MODEL_NAME = "facebook/nllb-200-distilled-600M"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
translation_pipeline = pipeline("translation", model=model, tokenizer=tokenizer, device=device, max_length=400, batch_size=16)

Using device: cuda


Device set to use cuda


In [46]:
from tqdm.auto import tqdm


def transformers_translator(
    df: pd.DataFrame,
    translation_pipe,
    text_col: str = "post_text",
    lang_col: str = "post_language",
    out_col: str = "translated_post",
    target_lang: str = "eng_Latn",
    pivot_lang: str = "spa_Latn",
    backtranslate_same_lang: bool = True,
) -> pd.DataFrame:
    """
    Translate df[text_col] → target_lang using NLLB pipeline.

    If src_lang == target_lang and backtranslate_same_lang=True,
    perform back-translation: target_lang → pivot_lang → target_lang.

    Args:
        df (pd.DataFrame):
            Input DataFrame containing text and language labels.
        translation_pipe (transformers.pipeline):
            Pre-loaded translation pipeline (e.g., NLLB-200).
        text_col (str):
            Column containing original text.
        lang_col (str):
            Column containing language codes matching the pipeline
            (e.g., "spa_Latn", "eng_Latn").
        out_col (str):
            Column name for the translated output.
        target_lang (str):
            Target language code (default: "eng_Latn").
        pivot_lang (str):
            Pivot language code for back-translation (default: "spa_Latn").
        backtranslate_same_lang (bool):
            If True, back-translate rows where src_lang == target_lang.

    Returns:
        pd.DataFrame:
            A DataFrame with the same index as df and one column (out_col).
    """

    result_df = pd.DataFrame(index=df.index)
    result_df[out_col] = None

    # Iterate per language with progress bar
    for src_lang, group in tqdm(df.groupby(lang_col), desc="Translating by language"):

        # Build a list[str]
        texts = group[text_col].astype(str).tolist()

        # Keep the original indexes so we can assign back later
        indexes = group.index.to_list()

        translated = []  # empty means "no translation done"

        # Case 1: normal translation src_lang -> target_lang
        if src_lang != target_lang:
            print(f"{len(texts)} posts: {src_lang} → {target_lang}")
            outputs = translation_pipe(
                texts,
                src_lang=src_lang,
                tgt_lang=target_lang,
            )
            translated = [o["translation_text"] for o in outputs]

        # Case 2: back-translation target_lang -> pivot_lang -> target_lang
        if (src_lang == target_lang) and backtranslate_same_lang:
            print(f"{len(texts)} posts: {target_lang} → {pivot_lang} → {target_lang}")
            pivot_outputs = translation_pipe(
                texts,
                src_lang=target_lang,
                tgt_lang=pivot_lang,
            )
            pivot_texts = [o["translation_text"] for o in pivot_outputs]

            back_outputs = translation_pipe(
                pivot_texts,
                src_lang=pivot_lang,
                tgt_lang=target_lang,
            )
            translated = [o["translation_text"] for o in back_outputs]

        if translated:
            result_df.loc[indexes, out_col] = translated

    return result_df

In [ ]:
translated_df = transformers_translator(df, translation_pipeline)
df = df.join(translated_df)

Translating by language:   0%|          | 0/40 [00:00<?, ?it/s]

162 posts: arb_Arab → eng_Latn
1 posts: bg → eng_Latn
118 posts: cat_Latn → eng_Latn
9 posts: cs → eng_Latn
30 posts: cym_Latn → eng_Latn
20 posts: da → eng_Latn
71 posts: deu_Latn → eng_Latn
9504 posts: eng_Latn → spa_Latn → eng_Latn


Your input_length: 400 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 400 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 399 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 399 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 400 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 366 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 399 is bigger than 0.9 * max_length: 400. You

In [ ]:
df[["post_text", "post_language", "translated_post"]].head()

In [ ]:
print(df['post_language'].value_counts())

# Remove Engllish Stopwords

In [ ]:
from nltk.corpus import stopwords
import nltk

In [ ]:
nltk.download('stopwords')

EXTRA_STOPWORDS = [
    'he', 'she', 'they', 'you', 'i', 'we', 'im', 'hes', 'shes', 'its', 'rt', 'amp', 're',
    'theyre', 'dont', 'doesnt', 'would', 'could', 'like', 'know', 'say', 'said', 'one',
    'look', 'looks', 'think', 'oh', 'yeah', 'guy', 'gonna', 'thing', 'someone', 'people',
    'get', 'got', 'thats', 'just', 'really', 'even', 'still', 'want', 'need', 'needs',
    'me', 'to', 'you', 'the', 'that', 'be', 'my', 'get', 'in', 'and',
     'rt', 'the', 'https', 'to', 'my', 'it', 'on', 'you', 'of', 'for', 'https', 'http', 'RT'
]
all_stopwords_english = set(stopwords.words('english') + EXTRA_STOPWORDS)

def remove_stopwords(text):
    return " ".join([w for w in text.lower().split() if w not in all_stopwords_english])


In [ ]:
df["clean_post_text"] = df["clean_post_text"].apply(remove_stopwords)

In [ ]:
df[["post_text", "clean_post_text"]].head(20)

# English Topic Clustering

## Utils


In [ ]:
def make_topic_stats(df, topic_col_name):
    """
    Calculate per-topic post counts.

    Args:
        df (pd.DataFrame): Data with topics and 'is_control'.
        topic_col_name (str): Column name of topic labels.

    Returns:
        pd.DataFrame: Topic stats with Count, IO_Count, Legitimate_posts.
    """
    topic_stats = (
        df.groupby(topic_col_name)
          .agg(
              Count=(topic_col_name, "count"),
              IO_Count=("is_control", lambda x: (~x).sum())
          )
          .reset_index()
    )
    topic_stats["Legitimate_posts"] = topic_stats["Count"] - topic_stats["IO_Count"]
    return topic_stats

In [ ]:
def plot_topic_stats(topic_stats, topic_col_name, model_name, output_path=""):
    """
    Plot stacked IO vs legitimate post counts per topic.
    Saves the fig to output_path if provided.

    Args:
        topic_stats (pd.DataFrame): Output of make_topic_stats().
        topic_col_name (str): Topic column to plot.

    Returns:
        None
    """
    topics = topic_stats[topic_col_name]
    io_count = topic_stats["IO_Count"]
    legitimate_count = topic_stats["Legitimate_posts"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    plt.title(f"Post Count per Topic, Model name: {model_name}")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.xticks(topics)
    plt.grid(axis="y")
    plt.legend()
    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=300) # save the fig

    plt.show()

In [ ]:
clean_posts_text_lst = df["clean_post_text"].to_list()

In [ ]:
from sklearn.preprocessing import normalize

# Load model
model = SentenceTransformer("digio/Twitter4SSE")

# Embeddings
embeddings = model.encode(clean_posts_text_lst, show_progress_bar=True)
embeddings = normalize(embeddings)

## BERTopic

In [ ]:
def english_BERTopic_modeling(posts_text_lst, embeddings, min_topic_size=10):
    """
    Run BERTopic using precomputed embeddings.

    Args:
        posts_text_lst (list[str]): English text posts.
        embeddings (np.ndarray): Precomputed embeddings aligned with the posts.
        min_topic_size (int): Minimum cluster size.

    Returns:
        topics, probs, topic_model
    """
    topic_model = BERTopic(
        language="english",
        min_topic_size=min_topic_size,
        verbose=True # show logs
    )

    topics, probs = topic_model.fit_transform(posts_text_lst, embeddings)

    return topics, probs, topic_model


In [ ]:
BERTopic_topic, BERT_probs, BERTopic_model = english_BERTopic_modeling(clean_posts_text_lst, embeddings)

df["BERTopic_topic"] = BERTopic_topic

BERTopic_model.get_topic_info()

In [ ]:
topic_col_name = "BERTopic_topic"
BERT_topic_stats = make_topic_stats(df, topic_col_name)
plot_topic_stats(BERT_topic_stats, topic_col_name, "BERTopic")

In [ ]:
from IPython.display import display

try:
    fig = BERTopic_model.visualize_topics(custom_labels=True)
    display(fig)   # <-- required to show it
except Exception as e:
    print("Not enough topics for visualization")
    print("Error:", e)

## K-Means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def auto_kmeans_topics(embeddings, k_min=4, k_max=40, step = 2):
    best_k = None
    best_score = -1
    best_labels = None

    for k in range(k_min, k_max + 1, step):
        km = KMeans(n_clusters=k, random_state=42, n_init="auto")
        labels = km.fit_predict(embeddings)
        score = silhouette_score(embeddings, labels)

        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels

    return best_labels, best_k

In [ ]:
# Add to df
labels, best_k = auto_kmeans_topics(embeddings)
df["K_Means_topic"] = labels
print(f"Best k: {best_k}")
df.head()

In [ ]:
topic_col_name = "K_Means_topic"
topic_stats = make_topic_stats(df, topic_col_name)
plot_topic_stats(topic_stats, topic_col_name, "K-Means")

# Named Entity Recognition (NER)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Load model ONCE
tokenizer = AutoTokenizer.from_pretrained("TweebankNLP/bertweet-tb2_wnut17-ner")
ner_model = AutoModelForTokenClassification.from_pretrained("TweebankNLP/bertweet-tb2_wnut17-ner")

ner_pipeline = pipeline(
    "token-classification",
    model=ner_model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)


def extract_ner_entities(posts_lst: list):
    """
    Runs NER in batches on a list of text posts.

    Args:
          posts_lst (list): List of strings (cleaned posts).
    Returns
    -------
    all_entities (list[list[str]]): For each post: a list of extracted entity words.
    """

    ner_outputs = ner_pipeline(posts_lst)
    all_entities = []
    for post_output in ner_outputs:
        words = [ent["word"] for ent in post_output]
        all_entities.append(words)

    return all_entities

In [ ]:
# Apply ONCE, no pandas apply, in order to utilize batches.
df["ner_entities"] = extract_ner_entities(df["clean_post_text"].to_list())

In [ ]:
df.loc[df["ner_entities"].astype(bool), ["clean_post_text", "ner_entities"]].head()

# Save to parquet

In [ ]:
df.to_parquet(output_folder_path, index=False, compression="gzip")

print(output_path)